# Raw Data Notebook

**Research question:** which characteristics of Steam games (genre, price, camera perspective, visual style,
narrative elements, gameplay features) are associated with higher player satisfaction and popularity?

This notebook loads the raw files our collectors produced and shows what they look like as collected,
including the problems that make them hard to use as-is. It does not clean or merge the data.

| Source | Collected by | Raw file | How it was obtained |
|---|---|---|---|
| SteamSpy catalog | `src/collect_steamspy.py ids` | `data/raw/steamspy_page_{0,10,30}.json` | REST API (`request=all`) |
| SteamSpy tags *(optional)* | `src/collect_steamspy.py details` | `data/raw/steamspy_raw.json` | REST API (`request=appdetails`) |
| Steam Store | `src/collect_games_data.py` | `data/raw/games_data.csv` | REST API (`store.steampowered.com/api/appdetails`) |
| Steam Reviews | `collect_reviews.py` | `data/steam_reviews_summary.csv` | REST API (`IUserReviewsService/GetAppReviews`) |

None of the raw files are modified.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
print("Project root:", ROOT)

Project root: /Users/shilpatel/successOnSteam


## 1. Load the raw sources

In [2]:
ids = pd.read_csv(RAW / "game_ids.csv")

catalog_pages = []
for page in (0, 10, 30):
    with open(RAW / f"steamspy_page_{page}.json", encoding="utf-8") as f:
        page_data = json.load(f)           # dict keyed by appid (as a string)
    df = pd.DataFrame(page_data.values())
    df["catalog_page"] = page
    catalog_pages.append(df)
catalog_raw = pd.concat(catalog_pages, ignore_index=True)

store_raw = pd.read_csv(RAW / "games_data.csv")

# The reviews CSV exists twice in the repo (root and data/). Use data/ and confirm they agree.
reviews_raw = pd.read_csv(ROOT / "data" / "steam_reviews_summary.csv")
root_copy = ROOT / "steam_reviews_summary.csv"
if root_copy.exists():
    other = pd.read_csv(root_copy)
    same = other.drop(columns="collected_at").sort_values("appid").reset_index(drop=True).equals(
        reviews_raw.drop(columns="collected_at").sort_values("appid").reset_index(drop=True))
    print("Root copy of reviews CSV matches data/ copy:", same)

tags_path = RAW / "steamspy_raw.json"
HAVE_TAGS = tags_path.exists()

summary = pd.DataFrame({
    "rows": [len(ids), len(catalog_raw), len(store_raw), len(reviews_raw)],
    "columns": [ids.shape[1], catalog_raw.shape[1], store_raw.shape[1], reviews_raw.shape[1]],
}, index=["game_ids (our sample)", "SteamSpy catalog (3 pages)", "Store API", "Reviews API"])
display(summary)
print("SteamSpy tag file present:", HAVE_TAGS)

Root copy of reviews CSV matches data/ copy: True


,rows,columns
game_ids (our sample),200,3
SteamSpy catalog (3 pages),3000,18
Store API,200,9
Reviews API,200,8


SteamSpy tag file present: True


## 2. What's messy in the raw data

A quick look at the problems in the raw data that would need fixing before the sources could be joined and analysed.

In [3]:
sample_ids = ids["appid"].tolist()
cat_s = catalog_raw[catalog_raw["appid"].isin(sample_ids)]

print("SteamSpy: numbers stored as text")
display(cat_s[["appid", "name", "owners", "price", "initialprice", "discount", "score_rank", "userscore"]].head(5))
print("dtypes:", cat_s[["owners", "price", "initialprice", "discount", "score_rank"]].dtypes.astype(str).to_dict())

print("\nStore API: categories/genres packed into one string, dates as text")
display(store_raw[["appid", "name", "genres", "categories", "release_date"]].head(3))

print("\nStore API: games showing $0 that are NOT free-to-play (delisted or no price on the store)")
not_f2p = store_raw[(store_raw["final_price"] == 0) & ~store_raw["genres"].fillna("").str.contains("Free To Play")]
display(not_f2p[["appid", "name", "genres", "final_price"]])

print("\nStore API: untranslated category keys")
raw_keys = sorted({c for v in store_raw["categories"].dropna() for c in v.split("; ") if c.startswith("#")})
print(raw_keys)

print("\nReviews API: rating label replaced by a count for low-review games")
display(reviews_raw["review_score_desc"].value_counts().to_frame("games"))

print("\nMissing values per source:")
display(pd.DataFrame({
    "store": store_raw.isna().sum()[store_raw.isna().sum() > 0],
}).T if store_raw.isna().any().any() else "none")

SteamSpy: numbers stored as text


,appid,name,owners,price,initialprice,discount,score_rank,userscore
6,1063730,New World: Aeternum,"50,000,000 .. 100,000,000",5999,5999,0,,0
25,252490,Rust,"20,000,000 .. 50,000,000",3999,3999,0,,0
27,346110,ARK: Survival Evolved,"20,000,000 .. 50,000,000",1499,1499,0,,0
30,1491000,War Robots: Frontiers,"20,000,000 .. 50,000,000",0,0,0,,0
32,218620,PAYDAY 2,"20,000,000 .. 50,000,000",499,999,50,,0


dtypes: {'owners': 'str', 'price': 'str', 'initialprice': 'str', 'discount': 'str', 'score_rank': 'str'}

Store API: categories/genres packed into one string, dates as text


,appid,name,genres,categories,release_date
0,453480,Shadowverse CCG,Strategy; Free To Play,Single-player; Multi-player; PvP; Online PvP; ...,"Oct 27, 2016"
1,466240,Deceit,Action; Indie; Free To Play,Multi-player; PvP; Online PvP; Steam Achieveme...,"Mar 3, 2017"
2,252490,Rust,Action; Adventure; Indie; Massively Multiplaye...,Multi-player; MMO; PvP; Online PvP; Co-op; Onl...,"Feb 8, 2018"



Store API: games showing $0 that are NOT free-to-play (delisted or no price on the store)


,appid,name,genres,final_price
13,63380,Sniper Elite V2,Action,0.0
37,1063730,New World: Aeternum,Action; Adventure; Massively Multiplayer; RPG,0.0
45,17480,Command & Conquer™ Red Alert™ 3,Action; Strategy,0.0
50,1811260,EA SPORTS™ FIFA 23,Simulation; Sports,0.0
89,1894790,Stones Keeper: King Aurelius,Strategy,0.0
107,1042530,POWER,Massively Multiplayer; Simulation; Strategy,0.0
114,432260,Tiltagon,Action; Casual; Indie,0.0
137,1703080,F-15 Strike Eagle,Simulation,0.0
139,743550,COSMONAUT,Action; Adventure; Indie,0.0
142,699760,Triple Otakus Puzzle,Casual; Indie,0.0



Store API: untranslated category keys
['#category_contrast_controls', '#category_playable_at_your_own_pace']

Reviews API: rating label replaced by a count for low-review games


,games
review_score_desc,
Very Positive,81
Mixed,36
Mostly Positive,31
Positive,18
Overwhelmingly Positive,9
Mostly Negative,7
1 user reviews,7
3 user reviews,3
4 user reviews,2



Missing values per source:


,release_date
store,1
